In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image
import glob
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

class PotatoDiseaseDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir  # Dataset path
        self.transform = transform  # Transformations
        self.class_labels = {
            "Early_blight": 0, "Late_blight": 1, "healthy": 2,
        }

        # Get all image paths
        self.image_paths = []
        self.labels = []
        for class_name, label in self.class_labels.items():
            class_images = glob.glob(f"{root_dir}/{class_name}/*.jpg")  # Find all images
            self.image_paths.extend(class_images)
            self.labels.extend([label] * len(class_images))  # Assign labels

    def __len__(self):
        return len(self.image_paths)  # Total number of images

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]  # Get image path
        label = self.labels[idx]  # Get label

        # Load image using PIL
        image = Image.open(image_path)

        # Apply transformations (if any)
        if self.transform:
            image = self.transform(image)

        return image, label  # Return processed image & label

        from torch.utils.data import DataLoader

# Define transformations
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images
    transforms.RandomRotation(15),  # Rotate images randomly within ±15 degrees
    transforms.ColorJitter(brightness=0.2),  #زAdjust brightness randomly + مو مطلوب انا حطيته
    transforms.ToTensor(),  # Convert to tensor
])

# Validation and testing data typically don’t require augmentations, as we only evaluate the model performance on these sets.
# Instead, we apply basic transformations to prepare the images.
transform_valid_test = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images to 64x64
    transforms.ToTensor(),  # Convert to tensor
])

# Initialize dataset for Train
train_path = os.path.join(path, "لازم احط باث بس نسيت كيف اجيبه", "Train")
test_path = os.path.join(path, "لازم احط باث بس نسيت كيف اجيب", "Test")

train_dataset = PotatoDiseaseDataset(train_path, transform=transform)
test_dataset = PotatoDiseaseDataset(test_path, transform=transform_valid_test)

# Create DataLoader
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# Get a batch of training images
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")

# Display some sample images with their labels


# Function to display images with predicted labels
def show_predictions(model, dataloader, device, num_images=10):
    model.eval()  # Set to evaluation mode
    images, labels = next(iter(dataloader))  # Get a batch
    images, labels = images.to(device), labels.to(device)

    with torch.no_grad():  # Disable gradient computation
        outputs = model(images)
        predictions = outputs.argmax(dim=1)  # Get predicted class

    # 3  class names
    classes = ["Early_blight", "Late_blight", "healthy"]

    # Plot images with predictions
    fig, axes = plt.subplots(2, 5, figsize=(10, 5))
    for i, ax in enumerate(axes.flat[:num_images]):
        img = images[i]
        img = np.transpose(img.cpu().numpy(), (1, 2, 0))  # Convert to (H, W, C) # لو الصورة على GPU → نرجعها CPU

        ax.imshow(img)
        ax.set_title(f"Pred: {classes[predictions[i].item()]}\nTrue: {classes[labels[i].item()]}") # عنوان الصورة
        ax.axis("off")

    plt.show()


In [ ]:
# Write your code here
import torch.nn as nn
import torch

# Define the CNN Model
class CNNModel(nn.Module): # ي نموذج في PyTorch لازم يرث من nn.Module
    def __init__(self): # هذا الـ constructor , نعرّف داخله الطبقات
        super(CNNModel, self).__init__() # خطوة ضرورية لتهيئة النموذج بشكل صحيح

        # Convolutional Layers # ديزاين على كيفي
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1) # زياده اسوي مرتين واغير الاوتبوت فهمت التاسك متاخر ولكن هذي الفكره العامه

        # Activation
        self.relu = nn.ReLU()

        # Pooling Layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) # تأخذ أكبر قيمة من كل مربع 2×2

        # Fully Connected Layers
        self.fc1 = nn.Linear(64 * 4 * 4, 128) # 4*4 الأبعاد صارت Conv + Pool  لأن بعد 3 مرات من
        self.fc2 = nn.Linear(128, 10)  # 10 output classes for CIFAR-10

    def forward(self, x): # يحدد كيف البيانات تمشي داخل النموذج
        # Convolution + ReLU + Pooling
        x = self.pool(self.relu(self.conv1(x)))  # (Batch, 16, 16, 16)
        x = self.pool(self.relu(self.conv2(x)))  # (Batch, 32, 8, 8)
        x = self.pool(self.relu(self.conv3(x)))  # (Batch, 64, 4, 4)

        # Flatten
        x = x.view(x.size(0), -1)  # (Batch, 64*4*4) # يمكن في اخر لير يكون اصغر من 32 عشان كذا سوينا هذي الخطوه زي مثال 50 صوره

        # Fully Connected Layers
        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # Logits #

        return x


In [ ]:
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device): # model → نموذج CNN , criterion → Loss function (مثل CrossEntropyLoss),
    model.train()  # Set model to training mode يفعّل:Dropout , BatchNorm
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader): # نمر على كل batch
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass # هذه Logits (مو احتمالات)
        loss = criterion(outputs, labels)  # Compute loss # نحسب الخطأ

        optimizer.zero_grad()  # Reset gradients # تصفير الـ gradients القديمة
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights # هذه اللحظة اللي النموذج يتعلّم فيها فعليًا

        total_loss += loss.item() # يحوّل Tensor → رقم

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1) # Softmax هنا فقط للتقييم
        predictions = outputs.argmax(dim=1)  # Get class with highest probability # نختار الفئة ذات أعلى احتمال
        correct += (predictions == labels).sum().item() # نقارن التوقع مع الحقيقة ونحسب
        total += labels.size(0) # نضيف عدد العينات في هذا batch

    avg_loss = total_loss / len(dataloader) # متوسط الخسارة لكل epoch
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode # يعطّل
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy



In [ ]:
# Write your code here
import torch.optim as optim # نستورد optimizers

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel().to(device) # البيانات والنموذج لازم يكونون على نفس الجهاز

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities) # مناسبة لـ Multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer # نستخدم
num_epochs = 10 # Number of epochs # النموذج سيرى البيانات 10 مرات


# Lists to store metrics # تجهيز قوائم لتخزين النتائج
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

    import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:
# Write your code here
